In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
To-Do List — Tkinter GUI (single file, no external deps)

Features:
- Persistent JSON storage at ~/.todo/todos.json (shared with the CLI version)
- Add / Edit / Delete tasks
- Mark Done / Undone
- Friendly due dates: "today", "tomorrow", "+3d", "+2w", "+1m", "+1y", or "YYYY-MM-DD"
- Priority 1..5 with visual stars
- Tags (comma-separated)
- Search box (title/description/tags)
- Filters: All / Pending / Done, Overdue only, Due Today
- Sort by clicking on column headers
- Recurring tasks: daily / weekly / monthly (auto-creates next on completion)
- Import / Export CSV via file dialog

Run:
    python todo_gui.py
"""
from __future__ import annotations

import csv
import datetime as dt
import json
import os
import sys
import uuid
from typing import Any, Dict, List, Optional, Tuple

import tkinter as tk
from tkinter import ttk, messagebox, filedialog

# ---------- Storage & Utils ----------
APP_DIR = os.path.join(os.path.expanduser("~"), ".todo")
DB_PATH = os.path.join(APP_DIR, "todos.json")
DATE_FMT = "%Y-%m-%d"

def ensure_db() -> None:
    os.makedirs(APP_DIR, exist_ok=True)
    if not os.path.exists(DB_PATH):
        with open(DB_PATH, "w", encoding="utf-8") as f:
            json.dump({"tasks": [], "version": 1}, f, indent=2)

def load_db() -> Dict[str, Any]:
    ensure_db()
    with open(DB_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_db(db: Dict[str, Any]) -> None:
    os.makedirs(APP_DIR, exist_ok=True)
    with open(DB_PATH, "w", encoding="utf-8") as f:
        json.dump(db, f, indent=2, ensure_ascii=False)

def now_utc() -> str:
    return dt.datetime.utcnow().isoformat(timespec="seconds") + "Z"

def gen_id() -> str:
    return uuid.uuid4().hex[:8]

def parse_date(s: Optional[str]) -> Optional[str]:
    """
    Accepts:
      - None or "" -> None
      - YYYY-MM-DD -> same (validated)
      - today, tomorrow
      - +Nd (days), +Nw (weeks), +Nm (months≈30d), +Ny (years≈365d)
    Returns YYYY-MM-DD or None; raises ValueError if invalid.
    """
    if not s:
        return None
    s = s.strip().lower()
    today = dt.date.today()
    if s == "today":
        return today.strftime(DATE_FMT)
    if s == "tomorrow":
        return (today + dt.timedelta(days=1)).strftime(DATE_FMT)
    if s.startswith("+") and len(s) >= 3:
        try:
            num = int(s[1:-1])
            unit = s[-1]
            if unit == "d":
                d = today + dt.timedelta(days=num)
            elif unit == "w":
                d = today + dt.timedelta(weeks=num)
            elif unit == "m":
                d = today + dt.timedelta(days=30 * num)
            elif unit == "y":
                d = today + dt.timedelta(days=365 * num)
            else:
                raise ValueError
            return d.strftime(DATE_FMT)
        except Exception as e:
            raise ValueError(f"Invalid relative date: {s}") from e
    # Absolute
    try:
        dt.datetime.strptime(s, DATE_FMT)
        return s
    except ValueError as e:
        raise ValueError("Date must be YYYY-MM-DD, 'today', 'tomorrow', or +Nd/+Nw/+Nm/+Ny") from e

def next_recurrence_date(due: Optional[str], recur: str) -> Optional[str]:
    if not due:
        return None
    base = dt.datetime.strptime(due, DATE_FMT).date()
    if recur == "daily":
        return (base + dt.timedelta(days=1)).strftime(DATE_FMT)
    if recur == "weekly":
        return (base + dt.timedelta(weeks=1)).strftime(DATE_FMT)
    if recur == "monthly":
        return (base + dt.timedelta(days=30)).strftime(DATE_FMT)
    return None

def priority_badge(p: int) -> str:
    return {5:"★★★★★",4:"★★★★☆",3:"★★★☆☆",2:"★★☆☆☆",1:"★☆☆☆☆"}.get(int(p), "—")

def is_overdue(due: Optional[str], completed: bool) -> bool:
    if completed or not due:
        return False
    try:
        return dt.datetime.strptime(due, DATE_FMT).date() < dt.date.today()
    except Exception:
        return False

def normalize_tags(s: str) -> List[str]:
    if not s:
        return []
    return sorted(set([x.strip() for x in s.split(",") if x.strip()]))

# ---------- Dialogs ----------
class TaskDialog(tk.Toplevel):
    def __init__(self, master, title: str, task: Optional[Dict[str, Any]] = None):
        super().__init__(master)
        self.title(title)
        self.resizable(False, False)
        self.transient(master)
        self.grab_set()

        self.result: Optional[Dict[str, Any]] = None

        frm = ttk.Frame(self, padding=12)
        frm.grid(row=0, column=0, sticky="nsew")

        # Fields
        ttk.Label(frm, text="Title *").grid(row=0, column=0, sticky="w")
        self.title_var = tk.StringVar(value=(task or {}).get("title", ""))
        ttk.Entry(frm, textvariable=self.title_var, width=48).grid(row=0, column=1, columnspan=3, sticky="ew", pady=2)

        ttk.Label(frm, text="Description").grid(row=1, column=0, sticky="nw")
        self.desc = tk.Text(frm, width=48, height=5)
        self.desc.grid(row=1, column=1, columnspan=3, sticky="ew", pady=2)
        if task:
            self.desc.insert("1.0", task.get("description",""))

        ttk.Label(frm, text="Due").grid(row=2, column=0, sticky="w")
        self.due_var = tk.StringVar(value=(task or {}).get("due","") or "")
        due_entry = ttk.Entry(frm, textvariable=self.due_var, width=20)
        due_entry.grid(row=2, column=1, sticky="w", pady=2)
        ttk.Label(frm, text="(YYYY-MM-DD, today, +3d...)").grid(row=2, column=2, columnspan=2, sticky="w")

        ttk.Label(frm, text="Priority").grid(row=3, column=0, sticky="w")
        self.pri_var = tk.StringVar(value=str((task or {}).get("priority", 3)))
        pri = ttk.Combobox(frm, textvariable=self.pri_var, width=5, values=["1","2","3","4","5"], state="readonly")
        pri.grid(row=3, column=1, sticky="w", pady=2)

        ttk.Label(frm, text="Tags").grid(row=3, column=2, sticky="e")
        self.tags_var = tk.StringVar(value=",".join((task or {}).get("tags", [])))
        ttk.Entry(frm, textvariable=self.tags_var, width=20).grid(row=3, column=3, sticky="w")

        ttk.Label(frm, text="Recurrence").grid(row=4, column=0, sticky="w")
        self.recur_var = tk.StringVar(value=(task or {}).get("recurrence") or "")
        recur = ttk.Combobox(frm, textvariable=self.recur_var, width=10, values=["", "daily", "weekly", "monthly"], state="readonly")
        recur.grid(row=4, column=1, sticky="w", pady=2)

        # Buttons
        btns = ttk.Frame(frm)
        btns.grid(row=5, column=0, columnspan=4, sticky="e", pady=(8,0))
        ttk.Button(btns, text="Cancel", command=self.destroy).grid(row=0, column=0, padx=4)
        ttk.Button(btns, text="Save", command=self.on_save).grid(row=0, column=1)

        self.bind("<Return>", lambda e: self.on_save())
        self.bind("<Escape>", lambda e: self.destroy())

    def on_save(self):
        title = self.title_var.get().strip()
        if not title:
            messagebox.showerror("Missing title", "Please enter a title.")
            return
        due_str = self.due_var.get().strip()
        try:
            due = parse_date(due_str) if due_str else None
        except ValueError as e:
            messagebox.showerror("Invalid date", str(e))
            return
        try:
            p = int(self.pri_var.get())
            if not (1 <= p <= 5):
                raise ValueError
        except Exception:
            messagebox.showerror("Invalid priority", "Priority must be an integer from 1 to 5.")
            return

        self.result = {
            "title": title,
            "description": self.desc.get("1.0", "end").strip(),
            "due": due,
            "priority": p,
            "tags": normalize_tags(self.tags_var.get()),
            "recurrence": self.recur_var.get() or None,
        }
        self.destroy()

# ---------- Main App ----------
class TodoApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("To-Do List")
        self.geometry("900x540")
        self.minsize(780, 480)

        # Theme tweaks
        style = ttk.Style(self)
        if "vista" in style.theme_names():
            style.theme_use("vista")
        style.configure("Treeview", rowheight=26)

        # State
        self.db: Dict[str, Any] = load_db()
        self.tasks: List[Dict[str, Any]] = self.db.get("tasks", [])
        self.filter_status = tk.StringVar(value="all")   # all|pending|done
        self.filter_overdue = tk.BooleanVar(value=False)
        self.filter_due_today = tk.BooleanVar(value=False)
        self.search_var = tk.StringVar()
        self.sort_by = ("due", True)  # (col_key, ascending)

        # UI
        self._build_menu()
        self._build_toolbar()
        self._build_tree()
        self._bind_shortcuts()
        self.refresh_tree()

    # ----- UI Builders -----
    def _build_menu(self):
        m = tk.Menu(self)
        # File
        filem = tk.Menu(m, tearoff=0)
        filem.add_command(label="Import CSV...", command=self.import_csv)
        filem.add_command(label="Export CSV...", command=self.export_csv)
        filem.add_separator()
        filem.add_command(label="Exit", command=self.quit)
        m.add_cascade(label="File", menu=filem)
        # View
        viewm = tk.Menu(m, tearoff=0)
        statusm = tk.Menu(viewm, tearoff=0)
        statusm.add_radiobutton(label="All", variable=self.filter_status, value="all", command=self.refresh_tree)
        statusm.add_radiobutton(label="Pending", variable=self.filter_status, value="pending", command=self.refresh_tree)
        statusm.add_radiobutton(label="Done", variable=self.filter_status, value="done", command=self.refresh_tree)
        viewm.add_cascade(label="Status", menu=statusm)
        viewm.add_checkbutton(label="Overdue only", variable=self.filter_overdue, command=self.refresh_tree)
        viewm.add_checkbutton(label="Due today", variable=self.filter_due_today, command=self.refresh_tree)
        m.add_cascade(label="View", menu=viewm)
        # Help
        helpm = tk.Menu(m, tearoff=0)
        helpm.add_command(label="About", command=self.show_about)
        m.add_cascade(label="Help", menu=helpm)
        self.config(menu=m)

    def _build_toolbar(self):
        bar = ttk.Frame(self, padding=(10,6))
        bar.pack(side="top", fill="x")

        ttk.Label(bar, text="Search:").pack(side="left")
        ent = ttk.Entry(bar, textvariable=self.search_var, width=32)
        ent.pack(side="left", padx=(4,10))
        self.search_var.trace_add("write", lambda *_: self.refresh_tree())

        ttk.Button(bar, text="Add", command=self.add_task).pack(side="left")
        ttk.Button(bar, text="Edit", command=self.edit_task).pack(side="left", padx=(6,0))
        ttk.Button(bar, text="Delete", command=self.delete_task).pack(side="left", padx=(6,12))
        ttk.Separator(bar, orient="vertical").pack(side="left", fill="y", padx=6)
        ttk.Button(bar, text="Done ✓", command=self.mark_done).pack(side="left")
        ttk.Button(bar, text="Undone ↩", command=self.mark_undone).pack(side="left", padx=(6,0))
        ttk.Separator(bar, orient="vertical").pack(side="left", fill="y", padx=6)
        ttk.Button(bar, text="Refresh", command=self.refresh_tree).pack(side="left")

    def _build_tree(self):
        cols = ("id","title","due","priority","tags","status")
        self.tree = ttk.Treeview(self, columns=cols, show="headings", selectmode="browse")
        self.tree.pack(fill="both", expand=True, padx=10, pady=8)

        self.tree.heading("id", text="ID", command=lambda: self.sort_by_column("id"))
        self.tree.heading("title", text="Title", command=lambda: self.sort_by_column("title"))
        self.tree.heading("due", text="Due", command=lambda: self.sort_by_column("due"))
        self.tree.heading("priority", text="Priority", command=lambda: self.sort_by_column("priority"))
        self.tree.heading("tags", text="Tags", command=lambda: self.sort_by_column("tags"))
        self.tree.heading("status", text="Status", command=lambda: self.sort_by_column("status"))

        self.tree.column("id", width=80, anchor="w")
        self.tree.column("title", width=300, anchor="w")
        self.tree.column("due", width=100, anchor="center")
        self.tree.column("priority", width=110, anchor="center")
        self.tree.column("tags", width=160, anchor="w")
        self.tree.column("status", width=100, anchor="center")

        # Row style via tag
        self.tree.tag_configure("done", foreground="#6a6a6a")
        self.tree.tag_configure("overdue", foreground="#b00020")

        # Double-click to edit
        self.tree.bind("<Double-1>", lambda e: self.edit_task())

    def _bind_shortcuts(self):
        self.bind("<Control-n>", lambda e: self.add_task())
        self.bind("<Control-e>", lambda e: self.edit_task())
        self.bind("<Delete>", lambda e: self.delete_task())
        self.bind("<Control-d>", lambda e: self.mark_done())
        self.bind("<Control-u>", lambda e: self.mark_undone())
        self.bind("<F5>", lambda e: self.refresh_tree())

    # ----- Core actions -----
    def apply_filters(self, items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        q = self.search_var.get().strip().lower()

        def match(t: Dict[str, Any]) -> bool:
            # status
            if self.filter_status.get() == "pending" and t["completed"]:
                return False
            if self.filter_status.get() == "done" and not t["completed"]:
                return False
            if self.filter_overdue.get() and not is_overdue(t.get("due"), t["completed"]):
                return False
            if self.filter_due_today.get():
                today = dt.date.today().strftime(DATE_FMT)
                if t["completed"] or t.get("due") != today:
                    return False
            if q:
                hay = " ".join([
                    t["title"],
                    t.get("description", ""),
                    " ".join(t.get("tags", [])),
                ]).lower()
                if q not in hay:
                    return False
            return True

        # exclude archived
        items = [t for t in items if not t.get("archived", False)]
        return [t for t in items if match(t)]

    def sort_items(self, items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        key, asc = self.sort_by
        reverse = not asc

        def keyfunc(t: Dict[str, Any]):
            if key == "priority":
                return (t["completed"], -int(t.get("priority", 3)), t.get("due") or "9999-12-31", t["title"].lower())
            if key == "due":
                return (t["completed"], t.get("due") or "9999-12-31", t["title"].lower())
            if key == "title":
                return (t["completed"], t["title"].lower())
            if key == "status":
                return (not t["completed"],)  # pending then done
            if key == "tags":
                return (",".join(t.get("tags",[])).lower(),)
            if key == "id":
                return (t["id"],)
            return (t["title"].lower(),)

        return sorted(items, key=keyfunc, reverse=reverse)

    def refresh_tree(self):
        self.tree.delete(*self.tree.get_children())
        filtered = self.apply_filters(self.tasks)
        ordered = self.sort_items(filtered)

        for t in ordered:
            status = "DONE" if t["completed"] else ("OVERDUE" if is_overdue(t.get("due"), t["completed"]) else "PENDING")
            tags = ",".join(t.get("tags", [])) or "-"
            due = t.get("due") or "-"
            pri = priority_badge(int(t.get("priority",3)))
            row_tag = ()
            if t["completed"]:
                row_tag = ("done",)
            elif is_overdue(t.get("due"), t["completed"]):
                row_tag = ("overdue",)
            self.tree.insert("", "end", iid=t["id"], values=(t["id"], t["title"], due, pri, tags, status), tags=row_tag)

        # status bar title
        self.update_idletasks()
        self.title(f"To-Do List — {len(ordered)} item(s)  •  {len(self.tasks)} total")

    def current_task(self) -> Optional[Dict[str, Any]]:
        sel = self.tree.selection()
        if not sel:
            return None
        tid = sel[0]
        for t in self.tasks:
            if t["id"] == tid:
                return t
        return None

    def add_task(self):
        dlg = TaskDialog(self, "Add Task")
        self.wait_window(dlg)
        if not dlg.result:
            return
        t = {
            "id": gen_id(),
            "title": dlg.result["title"],
            "description": dlg.result["description"],
            "due": dlg.result["due"],
            "priority": dlg.result["priority"],
            "tags": dlg.result["tags"],
            "recurrence": dlg.result["recurrence"],
            "completed": False,
            "created_at": now_utc(),
            "updated_at": now_utc(),
            "completed_at": None,
            "archived": False,
        }
        self.tasks.append(t)
        self.db["tasks"] = self.tasks
        save_db(self.db)
        self.refresh_tree()

    def edit_task(self):
        t = self.current_task()
        if not t:
            messagebox.showinfo("Edit Task", "Please select a task to edit.")
            return
        dlg = TaskDialog(self, "Edit Task", t)
        self.wait_window(dlg)
        if not dlg.result:
            return
        t["title"] = dlg.result["title"]
        t["description"] = dlg.result["description"]
        t["due"] = dlg.result["due"]
        t["priority"] = dlg.result["priority"]
        t["tags"] = dlg.result["tags"]
        t["recurrence"] = dlg.result["recurrence"]
        t["updated_at"] = now_utc()
        save_db(self.db)
        self.refresh_tree()

    def delete_task(self):
        t = self.current_task()
        if not t:
            messagebox.showinfo("Delete Task", "Please select a task to delete.")
            return
        if messagebox.askyesno("Confirm Delete", f"Delete '{t['title']}'?\n(Use CLI to clear archived permanently)"):
            # Soft-delete -> archive
            t["archived"] = True
            t["updated_at"] = now_utc()
            save_db(self.db)
            self.refresh_tree()

    def mark_done(self):
        t = self.current_task()
        if not t:
            messagebox.showinfo("Mark Done", "Please select a task.")
            return
        if t["completed"]:
            messagebox.showinfo("Already done", "This task is already completed.")
            return
        t["completed"] = True
        t["completed_at"] = now_utc()
        t["updated_at"] = now_utc()

        # Handle recurrence
        recur = (t.get("recurrence") or "").lower()
        if recur in {"daily","weekly","monthly"}:
            next_due = next_recurrence_date(t.get("due"), recur)
            new_t = dict(t)
            new_t["id"] = gen_id()
            new_t["completed"] = False
            new_t["completed_at"] = None
            new_t["created_at"] = now_utc()
            new_t["updated_at"] = now_utc()
            new_t["due"] = next_due
            new_t["archived"] = False
            self.tasks.append(new_t)
            self.db["tasks"] = self.tasks

        save_db(self.db)
        self.refresh_tree()

    def mark_undone(self):
        t = self.current_task()
        if not t:
            messagebox.showinfo("Mark Undone", "Please select a task.")
            return
        if not t["completed"]:
            messagebox.showinfo("Already pending", "This task is already pending.")
            return
        t["completed"] = False
        t["completed_at"] = None
        t["updated_at"] = now_utc()
        save_db(self.db)
        self.refresh_tree()

    def sort_by_column(self, col_key: str):
        key, asc = self.sort_by
        if key == col_key:
            self.sort_by = (col_key, not asc)
        else:
            self.sort_by = (col_key, True)
        self.refresh_tree()

    # ----- Import / Export -----
    def export_csv(self):
        path = filedialog.asksaveasfilename(
            title="Export CSV",
            defaultextension=".csv",
            filetypes=[("CSV files","*.csv"), ("All files","*.*")]
        )
        if not path:
            return
        try:
            with open(path, "w", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=[
                    "id","title","description","due","priority","tags","recurrence",
                    "completed","created_at","updated_at","completed_at","archived"
                ])
                w.writeheader()
                for t in self.db.get("tasks", []):
                    row = dict(t)
                    row["tags"] = ",".join(t.get("tags", []))
                    w.writerow(row)
            messagebox.showinfo("Exported", f"Exported {len(self.db.get('tasks', []))} tasks.")
        except Exception as e:
            messagebox.showerror("Export failed", str(e))

    def import_csv(self):
        path = filedialog.askopenfilename(
            title="Import CSV",
            filetypes=[("CSV files","*.csv"), ("All files","*.*")]
        )
        if not path:
            return
        try:
            with open(path, "r", encoding="utf-8") as f:
                r = csv.DictReader(f)
                rows = list(r)
            added = 0
            for row in rows:
                t = {
                    "id": row.get("id") or gen_id(),
                    "title": (row.get("title") or "").strip() or "(untitled)",
                    "description": (row.get("description") or "").strip(),
                    "due": row.get("due") or None,
                    "priority": int(row.get("priority") or 3),
                    "tags": normalize_tags(row.get("tags") or ""),
                    "recurrence": (row.get("recurrence") or "").strip() or None,
                    "completed": (row.get("completed","").lower() in {"true","1","yes","y"}),
                    "created_at": row.get("created_at") or now_utc(),
                    "updated_at": row.get("updated_at") or now_utc(),
                    "completed_at": row.get("completed_at") or None,
                    "archived": (row.get("archived","").lower() in {"true","1","yes","y"}),
                }
                self.tasks.append(t)
                added += 1
            self.db["tasks"] = self.tasks
            save_db(self.db)
            self.refresh_tree()
            messagebox.showinfo("Imported", f"Imported {added} task(s).")
        except Exception as e:
            messagebox.showerror("Import failed", str(e))

    # ----- Misc -----
    def show_about(self):
        messagebox.showinfo(
            "About",
            "To-Do List GUI\n"
            "Single-file Tkinter app (no external libraries)\n"
            "Storage: ~/.todo/todos.json\n"
            "• Add / Edit / Delete / Done / Undone\n"
            "• Search, Filters, Sort, Recurring, CSV import/export\n"
        )

def main() -> int:
    try:
        app = TodoApp()
        app.mainloop()
        return 0
    except KeyboardInterrupt:
        return 130

if __name__ == "__main__":
    sys.exit(main())


SystemExit: 0